# Signalling-Bots on Colab

1. **Runtime → Change runtime type → GPU** (e.g. T4).
2. Add your GitHub token as a Colab secret: key icon in the left sidebar → name it `GITHUB_TOKEN` → turn on notebook access.
3. Run the cells in order. All seeds in one run are evolved together on the GPU (`vmap` over seeds), so 10 seeds cost about the same as 1.
4. The last cell commits `results/<experiment>/...` and the run log, and pushes them to GitHub. Then `git pull` on your Mac to get them.

GPU maths rounds slightly differently from the Mac CPU, so a seed run here won't match the same seed run locally. Use new seed numbers to keep them apart.

In [ ]:
!nvidia-smi -L
import jax
print("jax", jax.__version__, jax.devices())

In [ ]:
REPO = "https://github.com/migarak2908/Signalling-Bots"
BRANCH = "main"

import os
os.chdir("/content")
if not os.path.exists("Signalling-Bots"):
    !git clone -b {BRANCH} {REPO}
os.chdir("/content/Signalling-Bots")
!git pull
!git log --oneline -1

In [ ]:
# Experiments: 2col_comm, 2col_nocomm, 3col_comm, 3col_nocomm
EXPERIMENT = "3col_comm"
SEEDS = "10 11 12 13 14 15 16 17 18 19"
GENERATIONS = 3000

LOG = f"results/logs/{EXPERIMENT}_s{SEEDS.replace(' ', '-')}_g{GENERATIONS}_colab.log"
!mkdir -p results/logs
!python -u -m evolve {EXPERIMENT} --seeds {SEEDS} --generations {GENERATIONS} 2>&1 | tee {LOG}

In [ ]:
!python -m analysis.summarize_runs {EXPERIMENT}

In [ ]:
from pathlib import Path
from IPython.display import Image, display

for seed in SEEDS.split():
    run = Path("results") / EXPERIMENT / f"s{seed}_g{GENERATIONS}"
    for plot in ("fitness.png", "trajectories.png"):
        display(Image(str(run / plot), width=900))

In [ ]:
# Commit this experiment's results and log, and push them to GitHub.
import subprocess
from google.colab import userdata

TOKEN_SECRET = "GITHUB_TOKEN"  # name of the Colab secret holding your GitHub token
token = userdata.get(TOKEN_SECRET)

!git config user.name "migarak2908"
!git config user.email "migarak2908@gmail.com"
!git add results/{EXPERIMENT} {LOG}
!git commit -q -m "Add {EXPERIMENT} results from Colab (seeds {SEEDS}, {GENERATIONS} generations)"

url = f"https://{token}@github.com/migarak2908/Signalling-Bots.git"
for cmd in (["git", "pull", "--rebase", "-q", url, BRANCH], ["git", "push", url, f"HEAD:{BRANCH}"]):
    out = subprocess.run(cmd, capture_output=True, text=True)
    print((out.stdout + out.stderr).replace(token, "***"))
    if out.returncode != 0:
        break

In [ ]:
# Optional: also copy the results to Google Drive.
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/Signalling-Bots/results
!cp -r results/{EXPERIMENT} /content/drive/MyDrive/Signalling-Bots/results/